# SVM versus deployed student: controlled Colab benchmark

This targeted audit complements the production experiments. It does **not** rerun teachers, coalitions, or student training. It reloads the ten saved proposed-student checkpoints and refits only the SVM configuration that was already selected from validation data.

The benchmark reports test accuracy and macro-F1, batch-one CPU latency, full-test throughput, serialized artifact size, numeric model-state size, student parameter count, and SVM support-vector count. SVM analytical MACs are intentionally reported as not applicable because kernel evaluation is not directly equivalent to the Conv1D/Dense MAC convention used for the neural network.

Latency is classifier-only. The SVM consumes handcrafted features and the student consumes standardized raw windows; their feature/preprocessing costs are not included. Therefore, these measurements do not establish end-to-end device latency.

Use a **CPU runtime** in Colab. Run all cells in order. Results are written to Google Drive under `MyDrive/makd-har/results/svm-student-comparison-v1/`.


In [ ]:
# Run this before importing TensorFlow.
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['NUMEXPR_NUM_THREADS'] = '1'
os.environ['TF_NUM_INTRAOP_THREADS'] = '1'
os.environ['TF_NUM_INTEROP_THREADS'] = '1'

from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import json, shutil, sys, time, platform, hashlib
import numpy as np
import pandas as pd

DRIVE_ROOT = Path('/content/drive/MyDrive/makd-har')
DRIVE_PROJECT_ROOT = next(
    (p for p in (DRIVE_ROOT, DRIVE_ROOT/'revised_pipeline')
     if (p/'run_experiment.py').is_file() and (p/'makd_har').is_dir()),
    None,
)
assert DRIVE_PROJECT_ROOT is not None, 'Project code was not found under MyDrive/makd-har.'

PROJECT_ROOT = Path('/content/revised_pipeline')
if PROJECT_ROOT.exists(): shutil.rmtree(PROJECT_ROOT)
shutil.copytree(DRIVE_PROJECT_ROOT, PROJECT_ROOT,
                ignore=shutil.ignore_patterns('results','__pycache__','*.pyc'))
sys.path.insert(0, str(PROJECT_ROOT))

OUTPUT_ROOT = DRIVE_ROOT/'results'/'svm-student-comparison-v1'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print('Code:', DRIVE_PROJECT_ROOT)
print('Output:', OUTPUT_ROOT)


In [ ]:
import tensorflow as tf
import sklearn
import joblib
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score
from makd_har.data import load_uci_har, load_wisdm_v1

try:
    tf.config.set_visible_devices([], 'GPU')
except RuntimeError:
    pass
tf.config.threading.set_intra_op_parallelism_threads(1)
tf.config.threading.set_inter_op_parallelism_threads(1)

SEEDS = [11,23,37,53,71,89,107,131,167,197]
WARMUPS = 30
REPETITIONS = 300
BATCH_REPETITIONS = 10
RESUME_COMPLETED_DATASETS = True

def production_dir(dataset):
    base=DRIVE_ROOT/'results'/dataset
    candidates=[base/'production-v2', base/'production']
    valid=[]
    for p in candidates:
        count=sum((p/f'seed-{s}.json').is_file() for s in SEEDS)
        if count: valid.append((count,p))
    assert valid, f'No production result directory found for {dataset}.'
    count,p=max(valid,key=lambda item:(item[0], item[1].name=='production-v2'))
    assert count==10, f'{p} contains only {count}/10 seed JSON files.'
    return p

def load_record(p,seed=11):
    return json.loads((p/f'seed-{seed}.json').read_text())

def build_bundle(dataset,record):
    c=record['config']
    if dataset=='uci':
        import zipfile
        archive=DRIVE_ROOT/'resources'/'UCI HAR Dataset.zip'
        assert archive.is_file(), archive
        extracted=Path('/content/uci-har-data')
        if extracted.exists(): shutil.rmtree(extracted)
        extracted.mkdir()
        with zipfile.ZipFile(archive) as z: z.extractall(extracted)
        return load_uci_har(extracted,
            validation_fraction=c['validation_fraction'], split_seed=c['split_seed'])
    raw=DRIVE_ROOT/'resources'/'WISDM_ar_v1.1_raw.txt'
    assert raw.is_file(), raw
    return load_wisdm_v1(raw,
        validation_fraction=c['validation_fraction'],
        test_fraction=c['wisdm_test_fraction'], split_seed=c['split_seed'])

def svm_report(record):
    for key in ('methods','baselines'):
        candidate=record.get(key,{}).get('svm')
        if candidate: return candidate
    raise KeyError('SVM report not found in seed JSON.')

def student_model_path(prod,seed):
    candidates=[prod/f'seed-{seed}'/'models'/'proposed_shapley_student.keras',
                prod/f'seed-{seed}'/'models'/'proposed.keras']
    for p in candidates:
        if p.is_file(): return p
    matches=list((prod/f'seed-{seed}').rglob('*proposed*.keras'))
    assert len(matches)==1, f'Expected one proposed checkpoint for seed {seed}; found {matches}'
    return matches[0]

def timed(fn,warmups=WARMUPS,repetitions=REPETITIONS):
    for _ in range(warmups): fn()
    ms=[]
    for _ in range(repetitions):
        t=time.perf_counter_ns(); fn(); ms.append((time.perf_counter_ns()-t)/1e6)
    a=np.asarray(ms)
    return {'warmups':warmups,'repetitions':repetitions,'mean_ms':float(a.mean()),
            'median_ms':float(np.median(a)),'std_ms':float(a.std(ddof=1)),
            'p95_ms':float(np.percentile(a,95)),'min_ms':float(a.min()),'max_ms':float(a.max())}

def numeric_svm_bytes(model):
    names=('support_vectors_','dual_coef_','intercept_','classes_','n_support_','probA_','probB_')
    return int(sum(getattr(model,n).nbytes for n in names if hasattr(getattr(model,n,None),'nbytes')))

def numeric_student_bytes(model): return int(sum(w.numpy().nbytes for w in model.weights))

print('TensorFlow',tf.__version__,'scikit-learn',sklearn.__version__)
print('CPU devices:',tf.config.list_physical_devices('CPU'),'GPU devices:',tf.config.list_physical_devices('GPU'))


In [ ]:
def benchmark_dataset(dataset):
    prod=production_dir(dataset)
    out_dir=OUTPUT_ROOT/dataset
    out_dir.mkdir(parents=True,exist_ok=True)
    completed=out_dir/'benchmark.json'
    if RESUME_COMPLETED_DATASETS and completed.is_file():
        previous=json.loads(completed.read_text())
        if len(previous.get('rows',[]))==11:
            print(dataset,'already complete; using saved dataset-level benchmark')
            return previous['rows'],previous
    record=load_record(prod)
    bundle=build_bundle(dataset,record)
    saved=svm_report(record)
    params=saved['selected_params']
    svm=SVC(**params,cache_size=4096)
    svm.fit(bundle.features_train,bundle.y_train)  # no test-data selection or retuning
    svm_pred=svm.predict(bundle.features_test)

    svm_file=out_dir/'selected-svm.joblib'
    joblib.dump(svm,svm_file,compress=0)
    svm_one=bundle.features_test[:1]
    svm_latency=timed(lambda: svm.predict(svm_one))
    svm_batch=timed(lambda: svm.predict(bundle.features_test),warmups=1,repetitions=BATCH_REPETITIONS)
    rows=[{
        'dataset':bundle.name,'method':'SVM','seed':None,
        'accuracy':accuracy_score(bundle.y_test,svm_pred),
        'macro_f1':f1_score(bundle.y_test,svm_pred,average='macro',zero_division=0),
        'batch1_mean_ms':svm_latency['mean_ms'],'batch1_median_ms':svm_latency['median_ms'],
        'batch1_p95_ms':svm_latency['p95_ms'],
        'full_test_samples_per_s':len(bundle.y_test)/(svm_batch['mean_ms']/1000),
        'artifact_mb':svm_file.stat().st_size/2**20,'numeric_state_mb':numeric_svm_bytes(svm)/2**20,
        'trainable_parameters':None,'support_vectors':int(svm.support_vectors_.shape[0]),
        'input_features':int(svm.support_vectors_.shape[1]),'input_representation':'handcrafted features',
        'kernel':params['kernel'],
    }]
    details={'dataset':bundle.name,'production_dir':str(prod),'selected_svm_params':params,
             'svm_latency':svm_latency,'svm_full_test_timing':svm_batch,
             'students':[]}
    for seed in SEEDS:
        seed_record=load_record(prod,seed)
        path=student_model_path(prod,seed)
        model=tf.keras.models.load_model(path,compile=False)
        x=tf.convert_to_tensor(bundle.raw_test)
        x1=x[:1]
        student_latency=timed(lambda: model(x1,training=False).numpy())
        student_batch=timed(lambda: model(x,training=False).numpy(),warmups=1,repetitions=BATCH_REPETITIONS)
        pred=np.argmax(model(x,training=False).numpy(),axis=1)
        measured_accuracy=float(accuracy_score(bundle.y_test,pred))
        measured_f1=float(f1_score(bundle.y_test,pred,average='macro',zero_division=0))
        saved_test=seed_record['methods']['proposed']['test']
        assert abs(measured_accuracy-float(saved_test['accuracy']))<1e-6
        assert abs(measured_f1-float(saved_test['macro_f1']))<1e-6
        row={
            'dataset':bundle.name,'method':'Proposed student','seed':seed,
            'accuracy':measured_accuracy,
            'macro_f1':measured_f1,
            'batch1_mean_ms':student_latency['mean_ms'],'batch1_median_ms':student_latency['median_ms'],
            'batch1_p95_ms':student_latency['p95_ms'],
            'full_test_samples_per_s':len(bundle.y_test)/(student_batch['mean_ms']/1000),
            'artifact_mb':path.stat().st_size/2**20,'numeric_state_mb':numeric_student_bytes(model)/2**20,
            'trainable_parameters':int(model.count_params()),'support_vectors':None,
            'input_features':int(np.prod(bundle.raw_test.shape[1:])),
            'input_representation':'standardized raw sensor window','kernel':None,
        }
        rows.append(row)
        details['students'].append({'seed':seed,'path':str(path),'latency':student_latency,
                                    'full_test_timing':student_batch})
        tf.keras.backend.clear_session()
        print(dataset,'seed',seed,'done')
    payload={'environment':{'platform':platform.platform(),'python':sys.version,
              'tensorflow':tf.__version__,'scikit_learn':sklearn.__version__,
              'cpu_only':True,'threads':1,'warmups':WARMUPS,'repetitions':REPETITIONS,
              'batch_repetitions':BATCH_REPETITIONS},'details':details,'rows':rows,
              'notes':{
                 'selection':'SVM hyperparameters were read from existing production results and not retuned.',
                 'macs':'SVM MACs are not reported; kernel evaluations are not directly comparable to the neural MAC convention.',
                 'latency_scope':'Classifier-only timing; feature extraction and preprocessing are excluded, so this is not end-to-end device latency.',
                 'energy':'Energy was not measured.',
                 'scope':'Teachers, coalitions, and students were not retrained.'}}
    (out_dir/'benchmark.json').write_text(json.dumps(payload,indent=2))
    pd.DataFrame(rows).to_csv(out_dir/'per-model.csv',index=False)
    return rows,payload

all_rows=[]; all_payload={}
for dataset in ('uci','wisdm'):
    rows,payload=benchmark_dataset(dataset)
    all_rows.extend(rows); all_payload[dataset]=payload
print('Benchmark complete.')


In [ ]:
df=pd.DataFrame(all_rows)
metrics=['accuracy','macro_f1','batch1_mean_ms','batch1_median_ms','batch1_p95_ms',
         'full_test_samples_per_s','artifact_mb','numeric_state_mb','trainable_parameters','support_vectors']
summary=[]
for (dataset,method),g in df.groupby(['dataset','method'],sort=False):
    row={'dataset':dataset,'method':method,'runs':len(g)}
    for m in metrics:
        vals=pd.to_numeric(g[m],errors='coerce').dropna()
        row[m+'_mean']=vals.mean() if len(vals) else np.nan
        row[m+'_sd']=vals.std(ddof=1) if len(vals)>1 else np.nan
    summary.append(row)
summary=pd.DataFrame(summary)
summary.to_csv(OUTPUT_ROOT/'summary.csv',index=False)

table=summary[['dataset','method','runs','accuracy_mean','accuracy_sd','macro_f1_mean','macro_f1_sd',
               'batch1_mean_ms_mean','batch1_p95_ms_mean','full_test_samples_per_s_mean',
               'artifact_mb_mean','numeric_state_mb_mean','trainable_parameters_mean','support_vectors_mean']].copy()
table['analytical_macs']=table['method'].map({'SVM':'not applicable','Proposed student':'see audited production results'})
table['energy']='not measured'
table.to_csv(OUTPUT_ROOT/'manuscript-table.csv',index=False)
(OUTPUT_ROOT/'benchmark-all.json').write_text(json.dumps(all_payload,indent=2))
display(table)


In [ ]:
import matplotlib.pyplot as plt
fig,axes=plt.subplots(2,2,figsize=(11,7.2))
colors={'Proposed student':'#087E83','SVM':'#173650'}
datasets=list(summary.dataset.unique())
for ax,(metric,title,scale) in zip(axes.ravel(),[
    ('accuracy_mean','Accuracy','linear'),('macro_f1_mean','Macro-F1','linear'),
    ('batch1_mean_ms_mean','Batch-one CPU latency (ms)','log'),
    ('numeric_state_mb_mean','Numeric model state (MB)','log')]):
    x=np.arange(len(datasets)); width=.34
    for j,method in enumerate(['Proposed student','SVM']):
        vals=[]; errs=[]
        for d in datasets:
            r=summary[(summary.dataset==d)&(summary.method==method)]
            vals.append(float(r[metric].iloc[0]))
            sd=r[metric[:-5]+'_sd'].iloc[0]
            errs.append(0 if pd.isna(sd) else float(sd))
        ax.bar(x+(j-.5)*width,vals,width,label=method,color=colors[method],
               yerr=errs if method=='Proposed student' else None,capsize=3)
    ax.set_xticks(x,datasets); ax.set_title(title,fontweight='bold'); ax.set_yscale(scale)
    ax.grid(axis='y',alpha=.25); ax.set_axisbelow(True)
axes[0,0].legend(frameon=False,ncol=2,loc='lower center')
fig.suptitle('Controlled SVM versus deployed-student comparison',fontweight='bold',fontsize=15)
fig.text(.5,.01,'Same Colab CPU session per dataset; classifier-only timing; batch size 1; energy not measured.',ha='center',fontsize=9)
fig.tight_layout(rect=(0,.04,1,.95))
fig.savefig(OUTPUT_ROOT/'svm-vs-student-comparison.png',dpi=300,bbox_inches='tight')
plt.show()


## Interpretation rules

- Use the accuracy and macro-F1 values to state which classifier predicts better on the fixed test sets.
- Compare batch-one latency only from this notebook because both methods are timed in the same session and protocol.
- The latency comparison excludes handcrafted-feature generation and raw-window preprocessing; do not describe it as end-to-end device latency.
- Treat artifact size as format-dependent. Numeric-state size is a closer—but still model-family-dependent—storage comparison.
- Do not convert SVM support vectors into “parameters” or report SVM kernel work as neural-network MACs.
- Do not claim energy efficiency because this notebook does not measure energy.
- Retain the JSON files as the audit trail and use `manuscript-table.csv` for the paper after the results have been checked.


In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
archive=OUTPUT_ROOT/'svm-student-comparison-v1.zip'
with ZipFile(archive,'w',ZIP_DEFLATED) as z:
    for p in OUTPUT_ROOT.rglob('*'):
        if p.is_file() and p != archive:
            z.write(p,p.relative_to(OUTPUT_ROOT))
print('Saved:',archive)
